In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [2]:
metrics = {
    "jmh": "bal_acc",
    "VMs": "bal_acc",
    "AzFuncInvoke": "smape",
    "WSD_3k": "smape",
    "AIOps": "auprc",
    "WSD": "auprc"
}

tasks = {
    "jmh": "steady-state",
    "VMs": "steady-state",
    "AzFuncInvoke": "timeseries-forecasting",
    "WSD_3k": "timeseries-forecasting",
    "AIOps": "anomaly-detection",
    "WSD": "anomaly-detection"
}

In [3]:
feature_metrics_variants = []
for metrics_file in Path("results/eval/ablation-features").glob("reshaped_metrics*"):
    variant_id = metrics_file.stem.split("#")[-1]
    
    metrics_df = pd.read_csv(metrics_file, index_col=[0])
    dataset_name = variant_id.split("---")[0]
    mean_metric = metrics_df[(metrics_df["model"] == "Meta")][metrics[dataset_name]].median()
    if metrics[dataset_name] == "smape":
        mean_metric = -mean_metric
    feature_metrics_variants.append({
        "variant_id": variant_id,
        "params": variant_id.split("---")[-1],
        "dataset": dataset_name,
        "metric": metrics[dataset_name], 
        "fs_algo": metrics_file.stem.split("---")[1],
        "variant_mean_metric": mean_metric
    })
feature_metrics_variants = pd.DataFrame(feature_metrics_variants)
feature_metrics_variants

,variant_id,params,dataset,metric,fs_algo,variant_mean_metric
0,AIOps---SelectKBest---function=f_regr___k=30,function=f_regr___k=30,AIOps,auprc,SelectKBest,0.778509
1,AIOps---SelectKBest---function=mutual_info_reg...,function=mutual_info_regression___k=30,AIOps,auprc,SelectKBest,0.778509
2,AIOps---random---k=30,k=30,AIOps,auprc,random,0.770749
3,WSD---SelectKBest---function=f_regr___k=30,function=f_regr___k=30,WSD,auprc,SelectKBest,0.201893
4,WSD---SelectKBest---function=mutual_info_regre...,function=mutual_info_regression___k=30,WSD,auprc,SelectKBest,0.201893
5,WSD---random---k=30,k=30,WSD,auprc,random,0.173347
6,WSD_3k---SelectKBest---function=f_regr___k=30,function=f_regr___k=30,WSD_3k,smape,SelectKBest,-5.817249
7,WSD_3k---SelectKBest---function=mutual_info_re...,function=mutual_info_regression___k=30,WSD_3k,smape,SelectKBest,-5.908352
8,WSD_3k---random---k=30,k=30,WSD_3k,smape,random,-5.969518
9,AzFuncInvoke---SelectKBest---function=f_regr__...,function=f_regr___k=30,AzFuncInvoke,smape,SelectKBest,-25.645839


In [4]:
choosen_variants = []
for rankfile in Path("results/eval").glob("reshaped_metrics*"):
    metrics_df = pd.read_csv(rankfile, index_col=[0])
    dataset_name = rankfile.stem.split("___")[-1]
    mean_metric = metrics_df[(metrics_df["model"] == "Meta")][metrics[dataset_name]].median()
    if metrics[dataset_name] == "smape":
        mean_metric = -mean_metric
    choosen_variants.append({
        "dataset": dataset_name,
        "metric": metrics[dataset_name], 
        "original_metric": mean_metric
    })
choosen_variants = pd.DataFrame(choosen_variants)
choosen_variants

,dataset,metric,original_metric
0,jmh,bal_acc,0.965517
1,VMs,bal_acc,0.980681
2,AIOps,auprc,0.780043
3,WSD,auprc,0.184727
4,WSD_3k,smape,-5.837343
5,AzFuncInvoke,smape,-25.645839


In [6]:
def compute_wins_df(df, metric):
    pivoted_df = df.pivot(columns="model", values=metric)

    wins = []
    loss = []

    for model in pivoted_df.columns:
        if model == "Meta":
            continue

        model_vals = pivoted_df[model]
        x_vals = pivoted_df['Meta']

        if metric == 'smape':
            # Lower is better
            better_mask = x_vals < model_vals
            worse_mask = x_vals > model_vals
        else:
            # Higher is better
            better_mask = x_vals > model_vals
            worse_mask = x_vals < model_vals

        equal_mask = x_vals == model_vals

        # Calculate counts
        better, worse, equal, total = better_mask.sum(), worse_mask.sum(), equal_mask.sum(), len(pivoted_df)

        # Calculate mean differences
        better_mean_diff = (x_vals[better_mask] - model_vals[better_mask]).mean()
        worse_mean_diff = (x_vals[worse_mask] - model_vals[worse_mask]).mean()
        equal_mean_diff = (x_vals[equal_mask] - model_vals[equal_mask]).mean()

        wins.append(round(100 * better / total, 1))
        loss.append(round(100 * worse / total, 1))

    # print(pivoted_df.columns)

    # print([(w, l) for w, l in zip(wins, loss)])

    # return wins, loss
    return [round(a - b, 1) for a, b in zip(wins, loss)]

def compute_wins(variant, baseline, metric):
    variant_df = pd.read_csv(f"results/eval/ablation-features/reshaped_metrics#{variant}.csv", index_col=[0])
    baseline_df = pd.read_csv(f"results/eval/reshaped_metrics_{baseline}.csv", index_col=[0])

    wins_variant = compute_wins_df(variant_df, metric)
    wins_baseline = compute_wins_df(baseline_df, metric)

    return [round(a - b, 1) for a, b in zip(wins_variant, wins_baseline)]

def compute_wins_var(variant, metric):
    variant_df = pd.read_csv(f"results/eval/ablation-features/reshaped_metrics#{variant}.csv", index_col=[0])

    wins_variant = compute_wins_df(variant_df, metric)
    # print(variant)
    
    return np.array(wins_variant)

def compute_wins_bs(baseline, metric):
    baseline_df = pd.read_csv(f"results/eval/reshaped_metrics_{baseline}.csv", index_col=[0])

    wins_baseline = compute_wins_df(baseline_df, metric)
    
    return np.array(wins_baseline)

def format_algo(row):
    if row['fs_algo'] == "SelectKBest":
        return 'SelectKBest' + f" ({row['params'].split('___')[0].split('=')[-1]})"
    if row['fs_algo'] == "all":
        return "All Features"
    if row['fs_algo'] == "random":
        return "Random Selection"
    return row['fs_algo']

In [7]:
# =============== ABLATION STUDY FOR FEATURES ======================

# Step 1: Get the best variant per (dataset, algo)

feature_metrics_variants['task'] = feature_metrics_variants['dataset'].apply(lambda x: tasks[x])

feature_metrics_variants['scores'] = feature_metrics_variants.apply(
    lambda row: compute_wins_var(variant=row['variant_id'], metric=row['metric']),
    axis=1
)

feature_metrics_variants['wins_mean_diff'] = feature_metrics_variants.apply(
    lambda row: np.mean(compute_wins_var(variant=row['variant_id'], metric=row['metric']) - compute_wins_bs(baseline=f"{row['task']}___{row['dataset']}", metric=row['metric'])),
    axis=1
)

feature_metrics_variants['always_best'] = feature_metrics_variants.apply(
    lambda row: np.all(compute_wins_var(variant=row['variant_id'], metric=row['metric']) > 0),
    axis=1
)

# best_variants = feature_metrics_variants.sort_values('wins_var', ascending=False).groupby(['dataset', 'fs_algo']).first().reset_index()
best_variants = feature_metrics_variants

features_merge = best_variants.merge(choosen_variants, on=["dataset", "metric"])

features_merge["custom_fs_algo"] = features_merge.apply(lambda row: format_algo(row), axis = 1)

features_merge = features_merge[["task", "dataset", "custom_fs_algo", "wins_mean_diff"]].sort_values(by=["task", "dataset", "custom_fs_algo"])

features_merge = features_merge.rename(columns={
    'custom_fs_algo': 'Algorithm',
    'metric': 'Metric',
    'wins_mean_diff': 'Delta Wins',
})

# # mean_best_variants

print("Maximize the metric")
# Perform the pivot
pivoted_df = features_merge.pivot_table(
    index='Algorithm',
    columns=['task', 'dataset'],  # group columns by Task and Dataset
    values='Delta Wins'
)

pivoted_df = pivoted_df.rename(columns={
    'AzFuncInvoke': 'Azure'
})

# Optional: sort columns for readability
pivoted_df = pivoted_df.sort_index(axis=1, level=[0,1]).applymap(lambda x: '(-)' if pd.notnull(x) and abs(x) < 1 else x)

pivoted_df.to_latex("results/tex/ablation_features_k=30.tex", column_format="l|cc|cc|cc", escape=True, float_format="%.1f")

Maximize the metric


/tmp/ipykernel_27989/4146421323.py:52: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pivoted_df = pivoted_df.sort_index(axis=1, level=[0,1]).applymap(lambda x: '(-)' if pd.notnull(x) and abs(x) < 1 else x)


In [8]:
pivoted_df


task                                 anomaly-detection       steady-state  \
dataset                                          AIOps   WSD          VMs   
Algorithm                                                                   
All Features                                    -11.02   (-)          (-)   
Random Selection                                -11.08  1.14          (-)   
SelectKBest (f_regr)                               (-)   3.5          (-)   
SelectKBest (mutual_info_regression)             -1.38  7.48          (-)   

task                                           timeseries-forecasting  \
dataset                                    jmh                  Azure   
Algorithm                                                               
All Features                               (-)              -4.050000   
Random Selection                     -3.133333              -3.600000   
SelectKBest (f_regr)                 -5.833333              -3.016667   
SelectKBest (mutual_info_regression)       (-)              -1.866667   

task                                            
dataset                                 WSD_3k  
Algorithm                                       
All Features                               (-)  
Random Selection                         -2.55  
SelectKBest (f_regr)                 -2.383333  
SelectKBest (mutual_info_regression)     -4.75